In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn


In [2]:
train = pd.read_csv('../data/labels/train_sent_emo.csv')
manifest = pd.read_csv('../data/manifest.csv')

In [3]:
X_text = np.load("../cache/text_concat.npy")
X_audio = np.load("../cache/audio_wavlm_13L.npy").astype(np.float32).reshape(-1, 13, 1536)

In [4]:

tr = (manifest.split == "train").values
va = (manifest.split == "dev").values
te = (manifest.split == "test").values

Xt_train, Xa_train = torch.from_numpy(X_text[tr]), torch.from_numpy(X_audio[tr])
Xt_val,   Xa_val   = torch.from_numpy(X_text[va]), torch.from_numpy(X_audio[va])

In [5]:
EMOTIONS = {"neutral":0, "joy":1, "surprise":2, "anger":3, "sadness":4, "disgust":5, "fear":6}
manifest['emo_encoded'] = manifest["Emotion"].map(EMOTIONS)
Y_train = manifest[manifest.split == 'train'].emo_encoded.to_numpy()
Y_train = torch.from_numpy(Y_train)

Y_val = manifest[manifest.split == 'dev'].emo_encoded.to_numpy()
Y_val = torch.from_numpy(Y_val)

Y_test = manifest[manifest.split == 'test'].emo_encoded.to_numpy()
Y_test = torch.from_numpy(Y_test)

In [6]:
class EmotionClassifier(nn.Module):

    def __init__(self):
        super().__init__()

        self.layer_w = nn.Parameter(torch.zeros(13))
        self.network = nn.Sequential(
            nn.Linear(2304, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(.5),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(.5),

            nn.Linear(128, len(EMOTIONS))
        )

    def forward(self, x_text, x_audio):
        w = torch.softmax(self.layer_w, 0)
        pooled = (x_audio * w[None, :, None]).sum(1)
        return self.network(torch.cat([x_text, pooled], 1))


counts = torch.bincount(Y_train)
class_weights = (1.0 / counts.float().sqrt())
class_weights = class_weights / class_weights.sum() * len(EMOTIONS)

loss_fn = nn.CrossEntropyLoss(weight=class_weights)




In [7]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import f1_score


model = EmotionClassifier()

optimizer = torch.optim.AdamW([
    {"params": model.network.parameters(), "lr": 1e-4, "weight_decay": 1e-2},
    {"params": [model.layer_w], "lr": 1e-2, "weight_decay": 0.0},
])

#train_ds = TensorDataset(X_train, Y_train)
train_dl = DataLoader(TensorDataset(Xt_train, Xa_train, Y_train), batch_size=256, shuffle=True)

best_f1 = -1
patience = 50
patience_counter = 0
best_weights = {k: v.clone() for k, v in model.state_dict().items()}


best_val_loss, best_weights, stale = float("inf"), None, 0
patience = 10

for epoch in range(100):
    model.train()
    for xt, xa, yb in train_dl:
        logits = model(xt, xa)
        loss = loss_fn(logits, yb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()

    model.eval()
    with torch.no_grad():
        val_loss = loss_fn(model(Xt_val, Xa_val), Y_val).item()
        val_f1 = f1_score(Y_val, model(Xt_val, Xa_val).argmax(1), average="weighted")

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_weights = {k: v.clone() for k, v in model.state_dict().items()}
        stale = 0
        flag=''
    else:
        stale += 1
        flag = "*"
    print(f"epoch {epoch:3d}  val_loss {val_loss:.4f}  dev_wF1 {val_f1:.4f}{flag}")
    if stale >= patience:
        break

model.load_state_dict(best_weights)

epoch   0  val_loss 1.9065  dev_wF1 0.1615
epoch   1  val_loss 1.7202  dev_wF1 0.4197
epoch   2  val_loss 1.6909  dev_wF1 0.4245
epoch   3  val_loss 1.6297  dev_wF1 0.4501
epoch   4  val_loss 1.6167  dev_wF1 0.4509
epoch   5  val_loss 1.6049  dev_wF1 0.4512
epoch   6  val_loss 1.5829  dev_wF1 0.4724
epoch   7  val_loss 1.5882  dev_wF1 0.4554*
epoch   8  val_loss 1.5740  dev_wF1 0.4645*
epoch   9  val_loss 1.5568  dev_wF1 0.4502*
epoch  10  val_loss 1.5381  dev_wF1 0.4757
epoch  11  val_loss 1.5460  dev_wF1 0.4762
epoch  12  val_loss 1.5264  dev_wF1 0.4715*
epoch  13  val_loss 1.5598  dev_wF1 0.4421*
epoch  14  val_loss 1.5533  dev_wF1 0.4770
epoch  15  val_loss 1.5218  dev_wF1 0.4801
epoch  16  val_loss 1.5176  dev_wF1 0.4685*
epoch  17  val_loss 1.4860  dev_wF1 0.5005
epoch  18  val_loss 1.5409  dev_wF1 0.4565*
epoch  19  val_loss 1.4854  dev_wF1 0.5028
epoch  20  val_loss 1.4590  dev_wF1 0.4938*
epoch  21  val_loss 1.5167  dev_wF1 0.4832*
epoch  22  val_loss 1.4754  dev_wF1 0.4918*
e

<All keys matched successfully>

In [8]:

model.eval()
with torch.no_grad():
    tr_pred = model(Xt_train, Xa_train).argmax(1)
    va_pred = model(Xt_val, Xa_val).argmax(1)

print("train wF1", f1_score(Y_train, tr_pred, average="weighted"))
print("dev   wF1", f1_score(Y_val, va_pred, average="weighted"))

train wF1 0.6477366301704419
dev   wF1 0.5136878187383603


In [9]:
from sklearn.metrics import classification_report
print(classification_report(Y_val, va_pred, target_names=EMOTIONS, zero_division=0))

              precision    recall  f1-score   support

     neutral       0.66      0.60      0.63       469
         joy       0.45      0.41      0.43       163
    surprise       0.42      0.58      0.48       150
       anger       0.50      0.57      0.53       153
     sadness       0.41      0.30      0.35       111
     disgust       0.13      0.18      0.15        22
        fear       0.18      0.17      0.18        40

    accuracy                           0.51      1108
   macro avg       0.39      0.40      0.39      1108
weighted avg       0.52      0.51      0.51      1108



In [10]:
print(torch.softmax(model.layer_w, 0).round(decimals=3))

tensor([0.1080, 0.0700, 0.0490, 0.0850, 0.0530, 0.0240, 0.0300, 0.0830, 0.1210,
        0.2060, 0.0900, 0.0530, 0.0290], grad_fn=<RoundBackward1>)
